In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
# Load data
sessions_featured = pd.read_csv('../data_csv/4 sessions_featured.csv')
sessions_modeled = pd.read_csv('../data_csv/5 sessions_modeled_lgb.csv')

sessions = pd.concat([sessions_featured, sessions_modeled[['purchase_prob', 'segment']]], axis=1)
sessions.head(1)

,user_session,user_id,session_start,session_end,total_events,num_views,num_cart,num_purchase,num_remove,num_returns,...,day_of_week,session_duration_min,converted,view_depth,single_view,hesitation_rate,cart_abandoned,cart_to_purchase_rate,purchase_prob,segment
0,0000061d-f3e9-484b-8c73-e54f355032a3,539262914,2020-01-16 06:30:41+03:00,2020-01-16 06:30:41+03:00,1,1,0,0,0,0,...,Thursday,0.0,0,1.0,1,NaN,0,NaN,0.000033,Low Intent


In [6]:
sessions.columns

Index(['user_session', 'user_id', 'session_start', 'session_end',
       'total_events', 'num_views', 'num_cart', 'num_purchase', 'num_remove',
       'num_returns', 'unique_products', 'avg_price', 'max_price', 'hour',
       'day_of_week', 'session_duration_min', 'converted', 'view_depth',
       'single_view', 'hesitation_rate', 'cart_abandoned',
       'cart_to_purchase_rate', 'purchase_prob', 'segment'],
      dtype='object')

In [ ]:
# Isolate high intent groups
high_intent_c  = sessions[(sessions['segment'] == 'High Intent') & (sessions['converted'] == 1)]
high_intent_nc = sessions[(sessions['segment'] == 'High Intent') & (sessions['converted'] == 0)]

print(f"High Intent converted:      {len(high_intent_c):,}")
print(f"High Intent non-converting: {len(high_intent_nc):,}")
print(f"Total sessions: {len(sessions):,}")

High Intent converted:      141,022
High Intent non-converting: 312,160
Total sessions: 4,535,940


In [7]:
# Behavioral comparison
cols = ['num_views', 'num_cart', 'num_remove', 'unique_products',
        'session_duration_min', 'hesitation_rate', 'view_depth',
        'single_view', 'avg_price', 'max_price', 'total_events']

comparison = pd.DataFrame({
    'Converted':     high_intent_c[cols].mean(),
    'Not Converted': high_intent_nc[cols].mean()
})

comparison['Difference %'] = (
    (comparison['Converted'] - comparison['Not Converted'])
    / comparison['Not Converted'] * 100
).round(1)

print("=== Behavioral Comparison ===")
print(comparison)
print()

# Hour comparison
print("=== Hour Distribution ===")
hour_comparison = pd.DataFrame({
    'Converted': high_intent_c['hour'].value_counts(normalize=True).sort_index(),
    'Not Converted': high_intent_nc['hour'].value_counts(normalize=True).sort_index()
})
hour_comparison['Difference %'] = (
    (hour_comparison['Converted'] - hour_comparison['Not Converted'])
    / hour_comparison['Not Converted'] * 100
).round(1)
print(hour_comparison)
print()

# Day comparison
print("=== Day of Week Distribution ===")
day_comparison = pd.DataFrame({
    'Converted': high_intent_c['day_of_week'].value_counts(normalize=True),
    'Not Converted': high_intent_nc['day_of_week'].value_counts(normalize=True)
})
day_comparison['Difference %'] = (
    (day_comparison['Converted'] - day_comparison['Not Converted'])
    / day_comparison['Not Converted'] * 100
).round(1)
print(day_comparison)

=== Behavioral Comparison ===
                      Converted  Not Converted  Difference %
num_views              6.611919       7.287189          -9.3
num_cart               7.758172       8.485427          -8.6
num_remove             4.533215       4.820429          -6.0
unique_products       14.754606      13.167443          12.1
session_duration_min  35.363743      42.221599         -16.2
hesitation_rate        0.446877       0.425771           5.0
view_depth             4.100974       3.017287          35.9
single_view            0.166740       0.143872          15.9
avg_price              7.413736       7.120812           4.1
max_price             19.499004      19.252004           1.3
total_events          27.589780      20.593045          34.0

=== Hour Distribution ===
      Converted  Not Converted  Difference %
hour                                        
0      0.051517       0.061328         -16.0
1      0.017508       0.021527         -18.7
2      0.009566       0.011930 

In [9]:
# Save diagnosis results
comparison.to_csv('diagnosis_behavioral.csv')
hour_comparison.to_csv('diagnosis_hour.csv')
day_comparison.to_csv('diagnosis_day.csv')